# 📰 audeMAG — Extraction Texte & Images depuis les PDFs
**MIASHS — Université Paul-Valéry Montpellier**

---

## 🗺️ Mode d'emploi (lisez avant de lancer)

1. **Cellule 1** → Installe les librairies *(à faire une seule fois par session)*
2. **Cellule 2** → Vous choisissez comment fournir vos PDFs *(upload manuel ou Google Drive)*
3. **Cellule 3** → Lance l'extraction sur **un seul PDF** ou **tous les PDFs** d'un coup
4. **Cellule 4** → Télécharge les résultats en `.zip`

---
### 📁 Structure de sortie générée
```
output/
└── audemag#1avril2016/
    ├── texte/
    │   ├── complet.txt        ← tout le texte du magazine
    │   └── par_page/
    │       ├── page_01.txt
    │       └── ...
    ├── images/
    │   ├── page_01_img_01.jpg
    │   └── ...
    └── metadata.json          ← index images + infos PDF
```

In [ ]:
# ╔══════════════════════════════════════════╗
# ║  CELLULE 1 — Installation des librairies ║
# ║  👉 Lancez cette cellule EN PREMIER      ║
# ╚══════════════════════════════════════════╝

!pip install pdfplumber pypdf Pillow --quiet
print('✅ Librairies installées avec succès')

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELLULE 2 — Chargement de vos PDFs                     ║
# ║  👉 Choisissez UNE des deux options ci-dessous           ║
# ╚══════════════════════════════════════════════════════════╝

# ─────────────────────────────────────────────────────────────
# OPTION A — Upload direct depuis votre ordinateur
#            (fonctionne toujours, même sans Google Drive)
# ─────────────────────────────────────────────────────────────
from google.colab import files
import os

os.makedirs('/content/pdfs', exist_ok=True)

print('📂 Sélectionnez vos PDFs (vous pouvez en sélectionner plusieurs à la fois)')
uploaded = files.upload()

for nom_fichier, contenu in uploaded.items():
    chemin = f'/content/pdfs/{nom_fichier}'
    with open(chemin, 'wb') as f:
        f.write(contenu)
    print(f'  ✅ Uploadé : {nom_fichier} ({len(contenu)/1024:.0f} Ko)')

pdfs_disponibles = [f'/content/pdfs/{f}' for f in os.listdir('/content/pdfs') if f.endswith('.pdf')]
print(f'\n📋 {len(pdfs_disponibles)} PDF(s) prêt(s) pour l\'extraction :')
for p in pdfs_disponibles:
    print(f'   • {os.path.basename(p)}')

In [ ]:
# ─────────────────────────────────────────────────────────────
# OPTION B — Depuis Google Drive
#            Décommentez ce bloc si vos PDFs sont sur Drive
# ─────────────────────────────────────────────────────────────

# from google.colab import drive
# drive.mount('/content/drive')
#
# # 👇 Changez ce chemin pour pointer vers votre dossier de PDFs sur Drive
# DOSSIER_DRIVE = '/content/drive/MyDrive/Magazines AUDEMAG/'
#
# import os
# pdfs_disponibles = [
#     os.path.join(DOSSIER_DRIVE, f)
#     for f in os.listdir(DOSSIER_DRIVE)
#     if f.lower().endswith('.pdf')
# ]
# print(f'{len(pdfs_disponibles)} PDF(s) trouvés sur Drive')
# for p in pdfs_disponibles:
#     print(f'  • {os.path.basename(p)}')

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELLULE 3 — Moteur d'extraction (ne pas modifier)      ║
# ╚══════════════════════════════════════════════════════════╝

import io, json, os, re
from pathlib import Path
import pdfplumber
from PIL import Image
from pypdf import PdfReader

# ── Paramètres (modifiables si besoin) ──────────────────────
MIN_SIZE_PX  = 80    # ignorer images < 80px (icônes, filets)
MIN_SIZE_KB  = 5     # ignorer images < 5 Ko
IMAGE_QUALITY = 90   # qualité JPEG de sortie
OUTPUT_DIR   = Path('/content/output')
# ────────────────────────────────────────────────────────────


def clean_text(raw):
    if not raw:
        return ''
    text = re.sub(r'(?<=[A-ZÀ-Ü])\s(?=[A-ZÀ-Ü])', '', raw)
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()


def extract_text(pdf_path, output_dir):
    text_dir    = output_dir / 'texte'
    perpage_dir = text_dir / 'par_page'
    perpage_dir.mkdir(parents=True, exist_ok=True)
    pages_text, full_parts = {}, []

    with pdfplumber.open(pdf_path) as pdf:
        nb = len(pdf.pages)
        for i, page in enumerate(pdf.pages):
            n = i + 1
            cleaned = clean_text(page.extract_text() or '')
            pages_text[n] = cleaned
            (perpage_dir / f'page_{n:02d}.txt').write_text(
                f'=== PAGE {n} / {nb} ===\n\n{cleaned}\n', encoding='utf-8'
            )
            full_parts.append(f"\n{'='*50}\n PAGE {n}\n{'='*50}\n\n{cleaned}\n")

    full_text = '\n'.join(full_parts)
    (text_dir / 'complet.txt').write_text(full_text, encoding='utf-8')
    return pages_text, len(full_text)


def is_valid_image(data, min_px, min_kb):
    if len(data) / 1024 < min_kb:
        return False, None
    try:
        pil = Image.open(io.BytesIO(data))
        w, h = pil.size
    except Exception:
        return False, None
    if w < min_px or h < min_px:
        return False, None
    if max(w, h) / max(min(w, h), 1) > 15:
        return False, None
    return True, pil


def save_image(pil, path):
    if pil.mode in ('CMYK', 'L', 'P', 'RGBA'):
        pil = pil.convert('RGB')
    path = path.with_suffix('.jpg')
    pil.save(path, format='JPEG', quality=IMAGE_QUALITY, subsampling=0)
    return path


def extract_images(pdf_path, output_dir, min_px, min_kb):
    images_dir = output_dir / 'images'
    images_dir.mkdir(parents=True, exist_ok=True)
    reader  = PdfReader(str(pdf_path))
    index   = []
    total_seen = total_kept = 0

    for page_num in range(1, len(reader.pages) + 1):
        page = reader.pages[page_num - 1]
        try:
            page_images = list(page.images)
        except Exception:
            continue
        counter = 0
        for raw_img in page_images:
            total_seen += 1
            try:
                valid, pil = is_valid_image(raw_img.data, min_px, min_kb)
            except Exception:
                continue
            if not valid:
                continue
            counter     += 1
            total_kept  += 1
            w, h = pil.size
            out_path = save_image(pil, images_dir / f'page_{page_num:02d}_img_{counter:02d}')
            index.append({
                'fichier'        : out_path.name,
                'chemin'         : str(out_path.relative_to(output_dir)),
                'page'           : page_num,
                'index_sur_page' : counter,
                'largeur_px'     : w,
                'hauteur_px'     : h,
                'taille_ko'      : round(len(raw_img.data) / 1024, 1),
                'description_ia' : None,
            })
    return index, total_kept, total_seen


def traiter_un_pdf(pdf_path):
    pdf_path   = Path(pdf_path)
    output_dir = OUTPUT_DIR / pdf_path.stem
    output_dir.mkdir(parents=True, exist_ok=True)

    print(f'\n📄 {pdf_path.name}')
    print(f'   Dossier de sortie : {output_dir}')

    # Métadonnées
    reader = PdfReader(str(pdf_path))
    meta   = reader.metadata or {}
    nb_pages = len(reader.pages)
    print(f'   Pages : {nb_pages}')

    # Texte
    print('   ⏳ Extraction texte...')
    pages_text, nb_chars = extract_text(pdf_path, output_dir)
    print(f'   ✅ Texte : {nb_chars:,} caractères')

    # Images
    print('   ⏳ Extraction images...')
    import warnings
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        images_index, kept, seen = extract_images(
            pdf_path, output_dir, MIN_SIZE_PX, MIN_SIZE_KB
        )
    print(f'   ✅ Images : {kept} conservées / {seen} trouvées')

    # JSON
    summary = {
        'fichier_source' : pdf_path.name,
        'nb_pages'       : nb_pages,
        'nb_images'      : kept,
        'nb_chars_texte' : nb_chars,
        'images'         : images_index,
    }
    (output_dir / 'metadata.json').write_text(
        json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8'
    )
    return summary


print('✅ Moteur chargé — prêt pour la cellule suivante')

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELLULE 4 — 🚀 LANCEMENT DE L'EXTRACTION               ║
# ║                                                          ║
# ║  MODE A : traiter UN seul PDF                            ║
# ║  MODE B : traiter TOUS les PDFs uploadés                 ║
# ║                                                          ║
# ║  👉 Choisissez le mode en bas de la cellule              ║
# ╚══════════════════════════════════════════════════════════╝

# ── MODE A : un seul PDF ────────────────────────────────────
# Remplacez le nom du fichier par le vôtre (tel qu'il apparaît après l'upload)

MON_PDF = 'audemag#1avril2016.pdf'   # 👈 CHANGEZ JUSTE CE NOM

chemin_pdf = f'/content/pdfs/{MON_PDF}'
resultat   = traiter_un_pdf(chemin_pdf)

print(f"""
╔══════════════════════════════════════════╗
  ✅  Extraction terminée !
  📄  Pages    : {resultat['nb_pages']}
  📝  Caractères : {resultat['nb_chars_texte']:,}
  🖼️   Images   : {resultat['nb_images']}
  📁  Résultats : /content/output/{Path(chemin_pdf).stem}/
╚══════════════════════════════════════════╝
""")

In [ ]:
# ── MODE B : TOUS les PDFs uploadés d'un coup ───────────────
# Décommentez ce bloc pour traiter tous vos PDFs en batch

# import os
# from pathlib import Path
#
# tous_les_pdfs = sorted(Path('/content/pdfs').glob('*.pdf'))
# rapport = []
#
# print(f'📦 {len(tous_les_pdfs)} PDF(s) à traiter\n')
#
# for i, pdf in enumerate(tous_les_pdfs, 1):
#     print(f'[{i}/{len(tous_les_pdfs)}]', end=' ')
#     try:
#         res = traiter_un_pdf(pdf)
#         rapport.append({'pdf': pdf.name, 'statut': 'OK',
#                         'pages': res['nb_pages'], 'images': res['nb_images']})
#     except Exception as e:
#         print(f'  ❌ Erreur : {e}')
#         rapport.append({'pdf': pdf.name, 'statut': f'ERREUR: {e}'})
#
# # Résumé
# print('\n' + '='*55)
# print(f'{"PDF":<35} {"Pages":>6} {"Images":>7} {"Statut"}')
# print('='*55)
# for r in rapport:
#     print(f"{r['pdf']:<35} {r.get('pages','-'):>6} {r.get('images','-'):>7}  {r['statut']}")
#
# import json
# Path('/content/output/rapport_batch.json').write_text(
#     json.dumps(rapport, ensure_ascii=False, indent=2), encoding='utf-8'
# )
# print(f'\n✅ Rapport batch sauvegardé : /content/output/rapport_batch.json')

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELLULE 5 — 📦 Téléchargement des résultats en ZIP     ║
# ║  👉 Lancez après l'extraction pour récupérer les fichiers║
# ╚══════════════════════════════════════════════════════════╝

import shutil
from google.colab import files
from pathlib import Path

# Choisir ce qu'on télécharge :
# - 'tout'   → tout le dossier output/ (textes + images)
# - 'textes' → uniquement les fichiers texte
# - 'images' → uniquement les images

TELECHARGER = 'tout'   # 👈 changez en 'textes' ou 'images' si besoin

OUTPUT_DIR = Path('/content/output')

if TELECHARGER == 'tout':
    source = OUTPUT_DIR
    zip_name = 'audemag_extraction_complete'

elif TELECHARGER == 'textes':
    # Rassemble uniquement les .txt dans un dossier temporaire
    tmp = Path('/content/tmp_textes')
    tmp.mkdir(exist_ok=True)
    for f in OUTPUT_DIR.rglob('*.txt'):
        dest = tmp / f.relative_to(OUTPUT_DIR)
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(f, dest)
    source   = tmp
    zip_name = 'audemag_textes'

elif TELECHARGER == 'images':
    tmp = Path('/content/tmp_images')
    tmp.mkdir(exist_ok=True)
    for f in OUTPUT_DIR.rglob('*.jpg'):
        dest = tmp / f.relative_to(OUTPUT_DIR)
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(f, dest)
    source   = tmp
    zip_name = 'audemag_images'

zip_path = f'/content/{zip_name}'
shutil.make_archive(zip_path, 'zip', source)
zip_final = zip_path + '.zip'

taille_mo = Path(zip_final).stat().st_size / (1024 * 1024)
print(f'📦 ZIP créé : {zip_final} ({taille_mo:.1f} Mo)')
print('⬇️  Téléchargement...')
files.download(zip_final)

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELLULE 6 — 🔍 Aperçu rapide des résultats             ║
# ║  (optionnel, pour vérifier avant de télécharger)        ║
# ╚══════════════════════════════════════════════════════════╝

import json
from pathlib import Path
from IPython.display import display, Image as IPImage

OUTPUT_DIR = Path('/content/output')

# ── Afficher le texte de la page 2 ──────────────────────────
premiers_pdfs = sorted(OUTPUT_DIR.iterdir())
if premiers_pdfs:
    dossier = premiers_pdfs[0]
    print(f'📄 Aperçu : {dossier.name}\n')

    # Texte page 2
    page2 = dossier / 'texte' / 'par_page' / 'page_02.txt'
    if page2.exists():
        print('── Texte page 2 (extrait) ──')
        print(page2.read_text(encoding='utf-8')[:500])
        print('...')

    # Index images
    meta_file = dossier / 'metadata.json'
    if meta_file.exists():
        meta = json.loads(meta_file.read_text(encoding='utf-8'))
        print(f'\n── {meta["nb_images"]} images extraites ──')
        for img in meta['images'][:5]:
            print(f'  {img["fichier"]:35s} | page {img["page"]} | {img["largeur_px"]}x{img["hauteur_px"]}px | {img["taille_ko"]} Ko')
        if meta['nb_images'] > 5:
            print(f'  ... et {meta["nb_images"]-5} autres')

    # Afficher les 3 premières images
    images = sorted((dossier / 'images').glob('*.jpg'))[:3]
    if images:
        print(f'\n── Aperçu des 3 premières images ──')
        for img_path in images:
            print(f'  {img_path.name}')
            display(IPImage(str(img_path), width=400))